# 07 Dense Hybrid Retrieval Comparison

## 목적
Dense, Hybrid Weighted, Hybrid RRF를 BM25와 같은 평가셋에서 비교하기 위한 공통 retrieval evaluation harness를 확인한다.

현재 notebook은 Dense/Hybrid 성능 평가가 아니다. BM25 baseline이 새 harness에서 재현되는지 확인하고, 이후 method를 추가할 비교 구조를 고정한다.

실험 순서와 후보군은 `docs/RETRIEVAL_ABLATION_PLAN.md`를 따른다. 이 notebook은 Stage 2(Dense baseline)와 Stage 3(Hybrid retrieval)를 담당한다.


## Ablation Matrix

Dense/Hybrid 비교 전에 chunking winner와 평가셋 fingerprint를 고정한다. Upstage API는 이 notebook에서 사용하지 않는다. Solar Pro 3는 LLM이 필요한 rewrite/HyDE 제한 실험과 generation 단계에서만 사용한다.

| stage | candidates | note |
| --- | --- | --- |
| Dense | `BAAI/bge-m3`, `intfloat/multilingual-e5-large`, `intfloat/multilingual-e5-large-instruct` | in-memory exact cosine으로 먼저 비교 |
| Hybrid | BM25 + Dense RRF, BM25 + Dense Weighted alpha 0.3/0.5/0.7 | 같은 chunk corpus와 같은 eval set 사용 |
| Production candidate | Qdrant hybrid query | in-memory 비교에서 winner가 나온 뒤 검증 |

성능 개선 주장은 dev/test split, paired comparison, bootstrap 10,000회, 95% CI 이후에만 허용한다. latency_p50_ms, latency_p95_ms, index build time, memory 사용량도 함께 기록한다.


In [ ]:
from pathlib import Path

report_path = Path('../evals/reports/retrieval_harness_report.md')
result_path = Path('../evals/results/retrieval_experiment_bm25_results.jsonl')
chunks_path_alias = '<private parent_child_chunks report>'
dataset_path = Path('../evals/datasets/retrieval_eval_seed.jsonl')
{
    'report_exists': report_path.exists(),
    'result_exists': result_path.exists(),
    'chunks_path_alias': chunks_path_alias,
    'dataset_path': str(dataset_path),
}


## 정량 결과 확인

BM25 결과가 기존 baseline과 같은 metric을 유지하는지 확인한다. latency는 실행 환경에 따라 달라질 수 있으므로 recall/MRR/nDCG를 중심으로 본다.


In [ ]:
report = report_path.read_text(encoding='utf-8')
expected_metrics = {
    'Recall@1': '0.083333',
    'Recall@3': '0.250000',
    'Recall@5': '0.250000',
    'MRR': '0.152778',
    'nDCG@5': '0.120124',
}
expected_metrics


In [ ]:
{metric: value in report for metric, value in expected_metrics.items()}


## Public Output Gate

공개 결과에는 검색 본문, context text, private path가 없어야 한다.


In [ ]:
result_text = result_path.read_text(encoding='utf-8')
drive_markers = ['F' + ':', 'C' + ':']
{
    'contains_text_field': '"text"' in result_text,
    'contains_search_text': 'search_text' in result_text,
    'contains_context_text': 'context_text' in result_text,
    'contains_private_windows_path': any(marker in result_text for marker in drive_markers),
}


## 판정

현재 harness는 BM25 baseline을 공통 실험 형식으로 재현한다. 다음 단계에서 Dense와 Hybrid를 추가하되, dataset과 metric은 변경하지 않는다.
